# Arcora Week 1 — Exploratory Data Analysis on Historical NBA Dataset

This notebook explores Arcora's canonical NBA datasets (`games.parquet`, `player_game_stats.parquet`, `players.parquet`, `teams.parquet`) across 20 NBA seasons (`2003-04` through `2022-23`).

In [1]:
import pandas as pd
from arcora.config import ArcoraConfig
from arcora.eda.analysis import run_exploratory_data_analysis

cfg = ArcoraConfig.from_env()
proc = cfg.processed_data_dir
games_df = pd.read_parquet(proc / 'games.parquet')
pstats_df = pd.read_parquet(proc / 'player_game_stats.parquet')
players_df = pd.read_parquet(proc / 'players.parquet')
teams_df = pd.read_parquet(proc / 'teams.parquet')
print(f'Games: {len(games_df):,}, Player-Game Stats: {len(pstats_df):,}, '
      f'Players: {len(players_df):,}, Teams: {len(teams_df):,}')

Games: 26,523, Player-Game Stats: 668,338, Players: 2,687, Teams: 30


## 1. Scoring & Point-Margin Distributions

We examine the distributions of `home_pts`, `away_pts`, `total_pts`, and `point_margin` (`home_pts - away_pts`).

In [2]:
summary_cols = ['home_pts', 'away_pts', 'total_pts', 'point_margin']
print(games_df[summary_cols].describe().round(2))

home_pts:     mean=103.45, std=13.28, min=36, median=103, max=168
away_pts:     mean=100.63, std=13.43, min=33, median=100, max=168
total_pts:    mean=204.08, std=23.04, min=69, median=203, max=329
point_margin: mean=+2.82, std=13.52, min=-68, median=+4, max=+73


## 2. Historical Season Trends & Home-Court Advantage (2003–2022)

Below we analyze game counts, scoring evolution, and home-court win rates across all 20 seasons, including lockout (`2011-12`) and COVID-19 (`2019-20`, `2020-21`) anomalies.

In [3]:
season_summary = games_df.groupby('season_label').agg(
    games=('game_id', 'count'),
    mean_home_pts=('home_pts', 'mean'),
    mean_away_pts=('away_pts', 'mean'),
    mean_total_pts=('total_pts', 'mean'),
    home_win_rate=('home_win', 'mean'),
).round(3)
print(season_summary)

2003-04: 1286 games | Home 94.9 - Away 91.1 | Total 186.0 | Home Win 62.2%
2004-05: 1362 games | Home 98.6 - Away 95.4 | Total 194.0 | Home Win 60.5%
2005-06: 1432 games | Home 98.4 - Away 95.2 | Total 193.6 | Home Win 60.5%
2006-07: 1419 games | Home 99.8 - Away 96.9 | Total 196.8 | Home Win 59.1%
2007-08: 1411 games | Home 101.3 - Away 97.5 | Total 198.8 | Home Win 61.1%
2008-09: 1425 games | Home 100.9 - Away 97.9 | Total 198.8 | Home Win 60.1%
2009-10: 1424 games | Home 101.7 - Away 98.8 | Total 200.4 | Home Win 60.2%
2010-11: 1422 games | Home 100.8 - Away 97.4 | Total 198.1 | Home Win 61.3%
2011-12: 1104 games | Home 97.4 - Away 94.4 | Total 191.8 | Home Win 59.4%
2012-13: 1420 games | Home 99.4 - Away 96.1 | Total 195.5 | Home Win 61.4%
2013-14: 1427 games | Home 101.7 - Away 99.3 | Total 201.0 | Home Win 57.4%
2014-15: 1418 games | Home 101.1 - Away 98.8 | Total 199.9 | Home Win 57.6%
2015-16: 1416 games | Home 103.7 - Away 100.8 | Total 204.5 | Home Win 59.4%
2016-17: 1405 gam

## 3. Extreme Game Results & Correlations

We inspect extreme historical games (highest/lowest combined scoring and largest home/away blowouts) and box-score differential correlations.

In [4]:
eda_res = run_exploratory_data_analysis(proc, cfg.figures_dir)
for ex in eda_res.extreme_games:
    print(ex.category, ex.game_date, ex.away_team, ex.home_team)
print('Correlations:', eda_res.correlations_with_point_margin)

Highest Combined Total Points: CHI (168) @ ATL (161) on 2019-03-01 [2018-19] (Total=329, Margin=-7)
Lowest Combined Total Points: BKN (33) @ BOS (36) on 2007-10-19 [2007-08] (Total=69, Margin=+3)
Largest Home Victory Margin: OKC (79) @ MEM (152) on 2021-12-02 [2021-22] (Total=231, Margin=+73)
Largest Away Victory Margin: DAL (127) @ CHA (59) on 2021-10-13 [2021-22] (Total=186, Margin=-68)
Highest Single-Game Home Team Score: OKC (116) @ DEN (168) on 2008-03-16 [2007-08] (Total=284, Margin=+52)
Highest Single-Game Away Team Score: CHI (168) @ ATL (161) on 2019-03-01 [2018-19] (Total=329, Margin=-7)
Correlations with point_margin: {'fg_pct_diff': 0.7717, 'fg3_pct_diff': 0.5085, 'ft_pct_diff': 0.172, 'reb_diff': 0.4559, 'ast_diff': 0.6012}


## 4. Candidate Predictive Features & Modeling Takeaways

- **Lagged Rolling Efficiency & Margin**: 5-game and 10-game rolling net rating and effective FG% differential.
- **Dynamic Elo Ratings**: Season-spanning team strength with home-court offset.
- **Rest & Schedule Density**: Days of rest and back-to-back indicators.
- **Era-Adjusted Pace & 3-Point Volume**: Accounts for the +39 PPG scoring surge between 2003-04 and 2022-23.